In [1]:
import pandas as pd
import yfinance as yf

In [2]:
#INTITAL PORTFOLIO FILTER -
#1. needs to be company stock
#2. remove stocks not listed in us or canada
#3. remove stocks not in USD or Cad
#4. keep only stocks with avg volume>=5000 (oct 1,2025 to sep 30,2026)
#5. exclude months with <18 trading days when finding avg


#read the tickers
tickers = pd.read_csv("Tickers_Example.csv", header=None)
tickers = tickers.dropna()

eligible = []

for i in range(len(tickers)):
    symbol = str(tickers.iloc[i, 0])
    valid = True

    try:                                   
        stock = yf.Ticker(symbol)
        info = stock.info

        #company stock?
        if info.get("quoteType") == "EQUITY":
            valid = True
        else:
            valid = False

        #listed in US/CANADA?
        valid_listing = False
        if info.get("market") == "us_market":
            valid_listing = True
        if info.get("market") == "ca_market":
            valid_listing = True
        if  valid_listing == False:
            valid = False

        #currency in usd/cad?
        valid_currency = False
        if info.get("currency") == "USD":
            valid_currency = True
        if info.get("currency") == "CAD":
            valid_currency= True
        if valid_currency == False:
            valid = False

        #exclude stocks with avgv volume <=5000 (oct 1,2025 to sep 30,2026) and months with <18 trading days are excluded in avg
        if valid:
            hist = stock.history(start="2025-10-01", end="2026-10-01")
            hist.index = hist.index.strftime('%Y-%m-%d')
            dates = pd.to_datetime(hist.index)

            days_per_month = hist["Volume"].groupby([dates.year, dates.month]).count()
            volume_per_month = hist["Volume"].groupby([dates.year, dates.month]).sum()

            total_days = 0
            total_volume = 0
            for m in range(len(days_per_month)):
                if days_per_month.iloc[m] >= 18:
                    total_days = total_days + days_per_month.iloc[m]
                    total_volume = total_volume + volume_per_month.iloc[m]

            if total_days == 0:
                valid = False
            elif total_volume/total_days < 5000:
                valid = False

    except Exception:
        valid = False

    #ticker alr in list?
    already_added = False
    for j in range(len(eligible)):
        if eligible[j] == symbol:
            already_added = True

    if valid and already_added == False:
        eligible.append(symbol)

print("Eligible tickers:", eligible)

Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
